# Unified Baselines (B7-B10) on Same Protocol as BRACED (B11)

**Run AFTER the main BRACED notebook** in the same Jupyter kernel. Reuses:
`kg`, `env_execute`, `run_two_hop`, `test`, `parse_gold`, `extract_entities`,
`tok`, and configuration.

## Protocol (identical to B11 to ensure apples-to-apples)
- Cleaned PrimeKG (DedupKG with 824 dup-groups merged)
- Top-K = 25
- No Strict Formatter (retrieval-as-answer); compute F1 over retrieved set
- 764 BioHopR test
- Same micro/macro F1 + bucket analysis

## Baselines
- **B7 Naive RAG** — extract entities from question, 2-hop typed retrieval (no decompose)
- **B8 CoT + KG** — GPT-4o-mini decomposes; we run KG retrieval per sub-query
- **B9 ReAct + KG** — GPT-4o-mini ReAct loop emits actions; we execute on KG
- **B10 SFT-only** — load `/sft_adapter` (no GRPO), greedy generate, retrieve

Headline B11 (already measured): micro-F1=0.3745, macro-F1=0.6176


In [ ]:
!ls -la /workspace/outputs/rl_dqd_fresh/best/adapter_model.safetensors 2>/dev/null && echo "✓ best อยู่" || echo "✗ best หาย!"
!ls -la /workspace/outputs/rl_dqd_fresh/sft_adapter/adapter_model.safetensors 2>/dev/null && echo "✓ sft อยู่" || echo "✗ sft หาย!"

In [ ]:
!pip install -q openai pyahocorasick peft accelerate transformers bitsandbytes

In [ ]:
import ahocorasick
print("✓ ahocorasick OK")

In [ ]:
import os
bdir = '/workspace/outputs/baselines/'
if os.path.exists(bdir):
    print("ไฟล์ผล baselines:", os.listdir(bdir))
else:
    print("✗ ไม่มีโฟลเดอร์ baselines — ผลเก่าหาย ต้องรันซ้ำ")

In [ ]:
import json
b8 = json.load(open('/workspace/outputs/baselines/B8_cot_kg.json'))
print(f"B8 CoT+KG: micro={b8['micro_f1']:.4f} macro={b8['macro_f1']:.4f} "
      f"P={b8['precision']:.3f} R={b8['recall']:.3f}")

In [ ]:
"""Restore kernel state WITHOUT retraining.
Loads: KG + matcher + data + model + LoRA adapter (/best from training).
Skips: SFT + GRPO.

Paste these blocks as SEPARATE cells in a fresh notebook (or run as one cell).
"""

# ============================================================
# BLOCK 1 — Config + imports
# ============================================================
import os, sys, json, re, ast, time, random, math, pickle
from pathlib import Path
import numpy as np
import torch

sys.path.insert(0, '/workspace')
random.seed(0); np.random.seed(0); torch.manual_seed(0)

CFG = dict(
    BACKBONE   = 'Qwen/Qwen2.5-7B-Instruct',
    LOOKUPS    = '/workspace/data/processed/entity_lookups.pkl',
    RELATIONS  = '/workspace/data/processed/relation_index.pkl',
    SPLITS     = '/workspace/cache/biohopr_splits.pkl',
    OUT        = '/workspace/outputs/rl_dqd_fresh',
    TOP_K      = 25,
    GROUP_G    = 8,
    TEMP       = 1.3,
    MAX_NEW    = 160,
    W_FORMAT=0.1, W_STRUCT=0.4, W_OUTCOME=0.5,
    W_SQ1=0.4, W_SQ2=0.6,
    EVAL_N = 764,
)
ANSWER_TYPES = ['disease','drug','gene/protein','phenotype','effect/phenotype']
print('Config ready. CUDA:', torch.cuda.is_available())

# ============================================================
# BLOCK 2 — KG + Aho-Corasick matcher
# ============================================================
from primekg_dedup import DedupKG
kg = DedupKG.load(CFG['LOOKUPS'], CFG['RELATIONS'], verbose=True)

try:
    import ahocorasick
    A = ahocorasick.Automaton()
    for name, eid in kg.name_to_id.items():
        if len(name) >= 4:
            A.add_word(name.lower(), (name.lower(), eid))
    A.make_automaton(); HAVE_AC = True
    print('Aho-Corasick built:', len(kg.name_to_id), 'names')
except Exception as e:
    HAVE_AC = False; print('no ahocorasick:', e)

def extract_entities(text, max_e=3):
    t = ' ' + text.lower() + ' '
    hits = {}
    if HAVE_AC:
        for end, (name, eid) in A.iter(t):
            start = end - len(name) + 1
            if (t[start-1] in ' (),.;:?') and (t[end+1] in ' (),.;:?'):
                can = kg.get_canonical_id(eid)
                hits[can] = max(hits.get(can, 0), len(name))
    else:
        for name, eid in kg.name_to_id.items():
            if len(name) >= 4 and (' '+name+' ') in t:
                can = kg.get_canonical_id(eid); hits[can] = max(hits.get(can,0), len(name))
    return [e for e,_ in sorted(hits.items(), key=lambda kv: -kv[1])][:max_e]

# ============================================================
# BLOCK 3 — Data
# ============================================================
splits = pickle.load(open(CFG['SPLITS'],'rb'))
train_set, test_set = splits['train'], splits['test']
def parse_gold(raw):
    if isinstance(raw, list): return [str(x).strip() for x in raw]
    try: return [str(x).strip() for x in ast.literal_eval(str(raw).replace('array(','').rstrip(')'))]
    except: return [str(raw).strip()]
def pack(s):
    return dict(question=s.get('hop2_question', s.get('prompt','')),
                source=str(s.get('hop2','')), bridge=str(s.get('hop1','')),
                bridge_type=str(s.get('hop1_type','')).strip(),
                target_type=str(s.get('target_type','')).strip(),
                gold=parse_gold(s.get('answer')))
train = [pack(s) for s in train_set]
test  = [pack(s) for s in test_set]
print('train', len(train), 'test', len(test))

# ============================================================
# BLOCK 4 — Smart Environment (2-hop typed)
# ============================================================
def typed_neighbors(eid, ttype, cap=None):
    ns = kg.get_neighbors(eid, top_k=None, target_type=(ttype or None))
    ns = list(ns)
    return ns[:cap] if cap else ns

def run_two_hop(seed_ids, bridge_type, target_type, top_k):
    hop1 = set()
    for sid in seed_ids: hop1 |= set(typed_neighbors(sid, bridge_type))
    hop2 = set()
    for b in list(hop1)[:300]: hop2 |= set(typed_neighbors(b, target_type))
    hop2_top = list(hop2)[:top_k]
    return hop1, set(hop2_top), hop2

USE_SOURCE_FALLBACK = False

def env_execute(sub_queries, ex):
    sq1 = sub_queries[0] if sub_queries else ''
    seeds = extract_entities(sq1)
    if not seeds and USE_SOURCE_FALLBACK:
        sid = kg.lookup(ex['source'])
        if sid: seeds = [sid]
    hop1, hop2_top, hop2_full = run_two_hop(seeds, ex['bridge_type'],
                                            ex['target_type'], CFG['TOP_K'])
    pred_names = [kg.id_to_name.get(x,'?') for x in hop2_top]
    return dict(seeds=seeds, hop1=hop1, hop2_top=hop2_top,
                hop2_full=hop2_full, pred_names=pred_names)
print('Smart Env ready')

# ============================================================
# BLOCK 5 — Load Qwen + apply BEST adapter (no train!)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(CFG['BACKBONE'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'left'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(CFG['BACKBONE'], quantization_config=bnb,
                                            device_map='auto', torch_dtype=torch.bfloat16)
# Load the trained best adapter directly
model = PeftModel.from_pretrained(base, CFG['OUT']+'/best', adapter_name='best')
model.set_adapter('best')
model.eval()
print('✓ loaded /best adapter (no retraining needed)')

# ============================================================
# BLOCK 6 — Prompt + parser helpers
# ============================================================
SYS = ("You are a biomedical reasoning planner. Decompose the question into 1-2 "
       "natural-language sub-queries that retrieve evidence from a knowledge graph. "
       "Output ONLY JSON: [{\"sub_query\": \"...\"}].")

def build_prompt(question):
    msgs = [{'role':'system','content':SYS},
            {'role':'user','content':f'Question: {question}\nJSON:'}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_subqueries(text):
    if not text: return []
    try:
        seg = text[text.find('['):text.rfind(']')+1]
        arr = json.loads(seg); sqs=[]
        for d in arr:
            if isinstance(d, dict):
                v = d.get('sub_query') or d.get('subquery') or d.get('query') or ''
                if v.strip(): sqs.append(v.strip())
            elif isinstance(d, str) and d.strip(): sqs.append(d.strip())
        if sqs: return sqs[:3]
    except Exception: pass
    found = re.findall(r'"sub_?query"\s*:\s*"([^"]+)"', text)
    if found: return [f.strip() for f in found][:3]
    lines = [l.strip(' -*0123456789.').strip() for l in text.splitlines()]
    return [l for l in lines if len(l.split())>=3 and not l.startswith('{') and not l.startswith('[')][:3]
print('✓ prompts + parser ready')

# ============================================================
# BLOCK 7 — Set OpenAI key (only needed for B8/B9 baselines)
# ============================================================
from getpass import getpass
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI key: ')
from openai import OpenAI
oai = OpenAI()
_t = oai.chat.completions.create(model='gpt-4o-mini', max_tokens=1,
        messages=[{'role':'user','content':'hi'}])
print('✓ OpenAI key works')

print('\n=== KERNEL RESTORED. unified_baselines.ipynb is ready to run. ===')


In [ ]:
# เช็คว่า kernel ยังมี state ของ main notebook ไหม
print('kg' in dir(), 'model' in dir(), 'test' in dir(), 'env_execute' in dir())

In [ ]:
# Reuse already-loaded state. Sanity-check the imports we need.
import os, json, re, ast, time, torch, numpy as np
from pathlib import Path
from openai import OpenAI
oai = OpenAI()

assert 'kg' in dir() and 'env_execute' in dir() and 'test' in dir(), \
    'Load the main BRACED notebook FIRST so kg/env/test are in kernel.'

OUTB = Path(CFG['OUT']).parent / 'baselines'
OUTB.mkdir(parents=True, exist_ok=True)
TOP_K = CFG['TOP_K']
EVAL_N = CFG['EVAL_N']

def f1_set(P, G):
    P={p.lower() for p in P}; G={g.lower() for g in G}
    if not P and not G: return 1.0,0,0,0
    if not P or not G:  return 0.0,0,0,len(G) if not P else len(P)
    tp=len(P&G); fp=len(P-G); fn=len(G-P)
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    return (2*pr*rc/(pr+rc) if pr+rc else 0, tp, fp, fn)

def bucket_report(per_q):
    buckets={'≤5':[], '6-15':[], '16-50':[], '>50':[]}
    for q in per_q:
        n=q['n_gold']
        k = '≤5' if n<=5 else '6-15' if n<=15 else '16-50' if n<=50 else '>50'
        buckets[k].append(q['f1'])
    return {k:(float(np.mean(v)) if v else 0.0, len(v)) for k,v in buckets.items()}

def aggregate(per_q):
    tp_g = sum(q['tp'] for q in per_q); fp_g = sum(q['fp'] for q in per_q); fn_g = sum(q['fn'] for q in per_q)
    pr = tp_g/(tp_g+fp_g) if tp_g+fp_g else 0
    rc = tp_g/(tp_g+fn_g) if tp_g+fn_g else 0
    micro = 2*pr*rc/(pr+rc) if pr+rc else 0
    macro = float(np.mean([q['f1'] for q in per_q]))
    return dict(micro_f1=micro, macro_f1=macro, precision=pr, recall=rc,
                n=len(per_q), bucket_f1=bucket_report(per_q))

def save(name, result, per_q):
    json.dump(result, open(OUTB/f'{name}.json','w'), indent=2)
    json.dump(per_q, open(OUTB/f'{name}_per_q.json','w'), indent=2)
    print(f"\n=== {name} on {result['n']} test ===")
    print(f"  micro-F1 = {result['micro_f1']:.4f}  (P={result['precision']:.3f}  R={result['recall']:.3f})")
    print(f"  macro-F1 = {result['macro_f1']:.4f}")
    for k,(f,n) in result['bucket_f1'].items():
        print(f"    {k:<8} n={n:>4}  F1={f:.4f}")

print('Helpers ready. EVAL_N =', EVAL_N, 'TOP_K =', TOP_K)

In [ ]:
# verify /best adapter ยัง generate sub-queries ดี (1 ตัว ฟรี)
import torch
ex = test[0]
enc = tok(build_prompt(ex['question']), return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**enc, do_sample=False, max_new_tokens=160,
                         pad_token_id=tok.pad_token_id)
txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
sqs = parse_subqueries(txt)
envout = env_execute(sqs, ex)
print('RAW:', repr(txt))
print('SQs:', sqs)
print(f"pred={len(envout['pred_names'])} gold={len(ex['gold'])} seeds={len(envout['seeds'])}")

### Answer Generation Module

In [ ]:
"""Answer Generation Module — ตรง concept: sub-query + context -> LLM -> answer.

นี่คือ MAIN pipeline ตาม concept หลักของงาน (ทางเลือก C):
  - Main:     LLM สังเคราะห์คำตอบจาก sub-query + retrieved context
  - Ablation: retrieval-as-answer (retrieved set = คำตอบ)

ต่างจาก Strict Formatter เดิมที่ fail:
  - เดิม: "เลือก candidates ที่ตอบ" -> conservative -> recall ตาย
  - ใหม่: "สังเคราะห์คำตอบจาก context" -> ครอบคลุมกว่า
  - prompt ออกแบบให้ generate คำตอบหลายตัว (multi-answer aware)

Run AFTER restore_kernel_state (ต้องมี kg, test, env_execute, parse_subqueries,
build_prompt, model, tok)
"""
import os, json, re, torch
from openai import OpenAI
oai = OpenAI()

# ============================================================
# Answer generation prompt — สังเคราะห์ ไม่ใช่แค่กรอง
# ============================================================
GEN_PROMPT = (
    "You are a biomedical question answering system. You are given a multi-hop "
    "question, the sub-queries used to decompose it, and the evidence entities "
    "retrieved from a knowledge graph for each sub-query.\n\n"
    "Using the retrieved evidence, answer the question by listing ALL entities "
    "that satisfy it. The question may use singular phrasing (e.g., 'Name a drug') "
    "but you should list every entity supported by the evidence. "
    "Base your answer on the retrieved evidence; these entities come from a curated "
    "biomedical knowledge graph.\n\n"
    "Question: {question}\n"
    "Sub-queries: {subqueries}\n"
    "Retrieved evidence: {context}\n\n"
    "Answer (comma-separated entity names):"
)

def generate_answer(question, subqueries, retrieved_names):
    """Main pipeline: sub-query + context -> LLM -> synthesized answer."""
    if not retrieved_names:
        return []
    ctx = ', '.join(retrieved_names)
    sq = '; '.join(subqueries) if subqueries else '(none)'
    prompt = GEN_PROMPT.format(question=question, subqueries=sq, context=ctx)
    try:
        r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0,
            max_tokens=500, messages=[{'role':'user','content':prompt}])
        txt = r.choices[0].message.content.strip()
        if txt.upper().startswith('NONE') or not txt:
            return []
        return [x.strip() for x in re.split(r'[,;\n]', txt) if x.strip()]
    except Exception as e:
        print('gen error:', e)
        return []

def f1_set(P, G):
    P={p.lower() for p in P}; G={g.lower() for g in G}
    if not P and not G: return 1.0,0,0,0
    if not P or not G:  return 0.0,0,0,(len(G) if not P else len(P))
    tp=len(P&G); fp=len(P-G); fn=len(G-P)
    pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    return (2*pr*rc/(pr+rc) if pr+rc else 0, tp, fp, fn)

# ============================================================
# DIAGNOSTIC: เทียบ main (LLM generate) vs ablation (retrieval) บน 20 test
# รันอันนี้ก่อนตัดสินใจ full eval
# ============================================================
def diagnose_generate_vs_retrieval(n=20):
    model.eval()
    rows = []
    gen_tp=gen_fp=gen_fn=0
    ret_tp=ret_fp=ret_fn=0
    print(f"Comparing LLM-generate (concept) vs retrieval-as-answer (ablation) on {n} test\n")
    for i in range(n):
        ex = test[i]
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, max_new_tokens=160,
                                 pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(txt)
        envout = env_execute(sqs, ex)
        retrieved = envout['pred_names']

        # MAIN: LLM generate
        gen_pred = generate_answer(ex['question'], sqs, retrieved)
        gf1, gtp, gfp, gfn = f1_set(gen_pred, ex['gold'])
        gen_tp+=gtp; gen_fp+=gfp; gen_fn+=gfn

        # ABLATION: retrieval-as-answer
        rf1, rtp, rfp, rfn = f1_set(retrieved, ex['gold'])
        ret_tp+=rtp; ret_fp+=rfp; ret_fn+=rfn

        rows.append((i, len(retrieved), len(gen_pred), len(ex['gold']), gf1, rf1))
        print(f"  t{i}: ctx={len(retrieved):2d} gen={len(gen_pred):2d} gold={len(ex['gold']):2d} "
              f"| gen-F1={gf1:.3f}  ret-F1={rf1:.3f}")

    def micro(tp,fp,fn):
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        return 2*pr*rc/(pr+rc) if pr+rc else 0, pr, rc
    gm, gp, gr = micro(gen_tp, gen_fp, gen_fn)
    rm, rp, rr = micro(ret_tp, ret_fp, ret_fn)
    print("\n" + "="*60)
    print(f"{'Pipeline':<28}{'micro-F1':>10}{'P':>8}{'R':>8}")
    print("-"*54)
    print(f"{'LLM generate (concept) ★':<28}{gm:>10.4f}{gp:>8.3f}{gr:>8.3f}")
    print(f"{'retrieval-as-answer (abl)':<28}{rm:>10.4f}{rp:>8.3f}{rr:>8.3f}")
    print("="*60)
    print("\nถ้า gen-F1 ใกล้ ret-F1 -> LLM generate ใช้ได้ (ตรง concept + F1 ดี)")
    print("ถ้า gen-F1 << ret-F1 -> narrow gold ยังกระทบ -> ปรับ prompt อีก")
    return rows

# diagnose_generate_vs_retrieval(20)   # <- รันอันนี้ก่อน


## Diagnose with Leakage

In [ ]:
"""วัด parametric leakage rate ของ LLM-generate pipeline.

Leakage = คำตอบที่ LLM ผลิต แต่ไม่ได้อยู่ใน retrieved candidates
        = LLM "เติมความรู้จากภายนอก" แทนที่จะยึด evidence

รายงาน 3 metric:
  1. F1 (LLM generate)        — main metric ตาม concept
  2. F1 (retrieval-as-answer) — ablation
  3. Leakage rate             — % ของ predicted entities ที่ไม่อยู่ใน context
                               + grounded F1 (F1 ของเฉพาะส่วนที่ยึด evidence)

Run AFTER answer_generation_module (ต้องมี generate_answer, f1_set, env_execute)
"""
import torch, re

def normalize_set(names):
    return {n.lower().strip() for n in names if n and n.strip()}

def diagnose_with_leakage(n=30):
    model.eval()
    gen_tp=gen_fp=gen_fn=0
    ret_tp=ret_fp=ret_fn=0
    grnd_tp=grnd_fp=grnd_fn=0   # grounded = เฉพาะคำตอบที่อยู่ใน context

    total_pred = 0
    total_leaked = 0            # predicted แต่ไม่อยู่ใน retrieved context
    n_with_leak = 0             # จำนวนคำถามที่มี leakage อย่างน้อย 1

    print(f"Measuring leakage on {n} test (gen vs retrieval + leakage rate)\n")
    print(f"{'#':>3} {'ctx':>4} {'gen':>4} {'leak':>4} {'gold':>5} {'genF1':>7} {'retF1':>7} {'grndF1':>7}")
    print("-"*48)

    for i in range(n):
        ex = test[i]
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, max_new_tokens=160,
                                 pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(txt)
        envout = env_execute(sqs, ex)
        retrieved = envout['pred_names']
        ctx_set = normalize_set(retrieved)

        gen_pred = generate_answer(ex['question'], sqs, retrieved)
        gen_set = normalize_set(gen_pred)
        gold_set = normalize_set(ex['gold'])

        # leakage: predicted ที่ไม่อยู่ใน retrieved context
        leaked = gen_set - ctx_set
        grounded = gen_set & ctx_set        # เฉพาะส่วนที่ยึด evidence
        total_pred += len(gen_set)
        total_leaked += len(leaked)
        if leaked: n_with_leak += 1

        # F1 ของแต่ละแบบ
        gf1, gtp, gfp, gfn = f1_set(gen_pred, ex['gold'])
        gen_tp+=gtp; gen_fp+=gfp; gen_fn+=gfn
        rf1, rtp, rfp, rfn = f1_set(retrieved, ex['gold'])
        ret_tp+=rtp; ret_fp+=rfp; ret_fn+=rfn
        # grounded F1: ใช้เฉพาะ predicted ที่อยู่ใน context
        gr_f1, grtp, grfp, grfn = f1_set(list(grounded), ex['gold'])
        grnd_tp+=grtp; grnd_fp+=grfp; grnd_fn+=grfn

        print(f"{i:>3} {len(retrieved):>4} {len(gen_set):>4} {len(leaked):>4} "
              f"{len(ex['gold']):>5} {gf1:>7.3f} {rf1:>7.3f} {gr_f1:>7.3f}")

    def micro(tp,fp,fn):
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        return (2*pr*rc/(pr+rc) if pr+rc else 0, pr, rc)
    gm,gp,gr = micro(gen_tp,gen_fp,gen_fn)
    rm,rp,rr = micro(ret_tp,ret_fp,ret_fn)
    grm,grp,grr = micro(grnd_tp,grnd_fp,grnd_fn)

    leak_rate = total_leaked/total_pred if total_pred else 0

    print("\n" + "="*60)
    print(f"{'Pipeline':<26}{'micro-F1':>10}{'P':>8}{'R':>8}")
    print("-"*52)
    print(f"{'LLM generate (concept) ★':<26}{gm:>10.4f}{gp:>8.3f}{gr:>8.3f}")
    print(f"{'retrieval-as-answer (abl)':<26}{rm:>10.4f}{rp:>8.3f}{rr:>8.3f}")
    print(f"{'LLM grounded-only':<26}{grm:>10.4f}{grp:>8.3f}{grr:>8.3f}")
    print("="*60)
    print(f"\n--- Parametric Leakage Analysis ---")
    print(f"  total predicted entities:     {total_pred}")
    print(f"  leaked (not in retrieved):    {total_leaked}")
    print(f"  ★ leakage rate:               {leak_rate:.1%}")
    print(f"  questions with any leakage:    {n_with_leak}/{n} ({n_with_leak/n:.1%})")
    print(f"\n  ตีความ:")
    print(f"  - leakage rate ต่ำ (<5%)  -> LLM ยึด evidence ดี, KG-grounding แข็งแรง")
    print(f"  - leakage rate กลาง (5-15%) -> มี leakage บ้าง, รายงานเป็น limitation")
    print(f"  - leakage rate สูง (>15%) -> LLM เติมเยอะ, ต้องคุม prompt เข้มขึ้น")
    return dict(gen_f1=gm, ret_f1=rm, grounded_f1=grm,
                leak_rate=leak_rate, n_with_leak=n_with_leak)

# result = diagnose_with_leakage(30)


## Grounded Eval Module

In [ ]:
"""Grounded-only evaluation — main metric สำหรับทุก baseline.

วัด 3 metric พร้อมกันในการรันครั้งเดียว (ประหยัด API):
  - grounded-only (MAIN): generate -> ตัดคำตอบที่ไม่อยู่ใน retrieved
  - generate raw (ablation 1): generate ตรงๆ (มี leakage)
  - retrieval-as-answer (ablation 2): retrieved = answer

ใช้กับทุก baseline: B7, B7b, B8, B9, B10, B11
Run AFTER answer_generation_module (ต้องมี generate_answer, f1_set)
"""
import torch

def _norm(names):
    return {n.lower().strip() for n in names if n and n.strip()}

def generate_answer_grounded(question, sqs, retrieved):
    """MAIN pipeline: generate + post-filter ให้ยึด retrieved KG 100%."""
    raw = generate_answer(question, sqs, retrieved)
    ctx = _norm(retrieved)
    grounded = [p for p in raw if p.lower().strip() in ctx]
    return grounded, raw   # คืนทั้ง grounded (main) และ raw (ablation)

def eval_baseline_3tier(get_subqueries_and_retrieved, name="baseline", n=None,
                         use_llm=True):
    """ประเมิน baseline แบบวัด 3 metric พร้อมกัน.

    get_subqueries_and_retrieved(ex) -> (sqs, retrieved_names)
        callback ที่แต่ละ baseline กำหนดเอง (วิธีได้ sub-query + retrieved ต่างกัน)
    use_llm: True = วัด grounded+raw+retrieval / False = retrieval อย่างเดียว
             (B7 Naive RAG อาจไม่ต้องใช้ LLM ก็ได้ แต่เพื่อ fair ใช้ True หมด)
    """
    n = n or len(test)
    acc = {'grounded':[0,0,0], 'raw':[0,0,0], 'retrieval':[0,0,0]}  # tp,fp,fn
    macro = {'grounded':[], 'raw':[], 'retrieval':[]}
    total_pred=total_leaked=n_leak=0

    print(f"[{name}] evaluating {n} questions (3-tier: grounded/raw/retrieval)...")
    for i in range(n):
        ex = test[i]
        sqs, retrieved = get_subqueries_and_retrieved(ex)

        # retrieval-as-answer
        rf1, rtp, rfp, rfn = f1_set(retrieved, ex['gold'])
        acc['retrieval'][0]+=rtp; acc['retrieval'][1]+=rfp; acc['retrieval'][2]+=rfn
        macro['retrieval'].append(rf1)

        if use_llm:
            grounded, raw = generate_answer_grounded(ex['question'], sqs, retrieved)
            # leakage tracking
            ctx=_norm(retrieved); rawset=_norm(raw)
            leaked=rawset-ctx
            total_pred+=len(rawset); total_leaked+=len(leaked)
            if leaked: n_leak+=1
            # raw
            raf1, ratp, rafp, rafn = f1_set(raw, ex['gold'])
            acc['raw'][0]+=ratp; acc['raw'][1]+=rafp; acc['raw'][2]+=rafn
            macro['raw'].append(raf1)
            # grounded (MAIN)
            gf1, gtp, gfp, gfn = f1_set(grounded, ex['gold'])
            acc['grounded'][0]+=gtp; acc['grounded'][1]+=gfp; acc['grounded'][2]+=gfn
            macro['grounded'].append(gf1)
        else:
            acc['grounded']=acc['retrieval']; acc['raw']=acc['retrieval']
            macro['grounded']=macro['retrieval']; macro['raw']=macro['retrieval']

        if (i+1)%100==0: print(f"  ...{i+1}/{n}")

    def micro(tpl):
        tp,fp,fn=tpl
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        return (2*pr*rc/(pr+rc) if pr+rc else 0, pr, rc)

    res={}
    for k in ['grounded','raw','retrieval']:
        mi,p,r=micro(acc[k]); ma=sum(macro[k])/len(macro[k]) if macro[k] else 0
        res[k]=dict(micro_f1=mi, macro_f1=ma, P=p, R=r)
    res['leak_rate']=total_leaked/total_pred if total_pred else 0
    res['n_leak']=n_leak; res['name']=name

    print(f"\n[{name}] results:")
    print(f"  {'tier':<22}{'micro-F1':>9}{'macro-F1':>9}{'P':>7}{'R':>7}")
    print(f"  {'grounded (MAIN) ★':<22}{res['grounded']['micro_f1']:>9.4f}"
          f"{res['grounded']['macro_f1']:>9.4f}{res['grounded']['P']:>7.3f}{res['grounded']['R']:>7.3f}")
    print(f"  {'generate raw':<22}{res['raw']['micro_f1']:>9.4f}"
          f"{res['raw']['macro_f1']:>9.4f}{res['raw']['P']:>7.3f}{res['raw']['R']:>7.3f}")
    print(f"  {'retrieval-as-answer':<22}{res['retrieval']['micro_f1']:>9.4f}"
          f"{res['retrieval']['macro_f1']:>9.4f}{res['retrieval']['P']:>7.3f}{res['retrieval']['R']:>7.3f}")
    if use_llm:
        print(f"  leakage rate: {res['leak_rate']:.1%} ({n_leak}/{n} questions)")
    return res

# ====== callback สำหรับ B11 BRACED (policy generate sub-queries) ======
def b11_get(ex):
    prompt = build_prompt(ex['question'])
    enc = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, max_new_tokens=160,
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sqs = parse_subqueries(txt)
    envout = env_execute(sqs, ex)
    return sqs, envout['pred_names']

# # ตัวอย่างรัน B11 ด้วย grounded main:
# model.eval()
# res_b11 = eval_baseline_3tier(b11_get, name="B11 BRACED", n=764)


In [ ]:
import inspect
print(inspect.getsource(env_execute))
print("test[0] keys:", list(test[0].keys()))
print("sample:", {k: (str(v)[:60]) for k,v in test[0].items()})

In [ ]:
# เช็ค method ของ kg
print([m for m in dir(kg) if not m.startswith('_')])
# เช็คว่า model มี adapter อะไรบ้าง
print(type(model))
print("adapters:", model.peft_config.keys() if hasattr(model,'peft_config') else "ไม่ใช่ PeftModel")

In [ ]:
"""PATCH: เพิ่ม grounded-only เป็น main metric ให้ทุก baseline.

วางเป็น cell ใหม่ "หลัง cell 10 (grounded module)" และ "ก่อน B7"
ใช้ helper เดิม (f1_set, aggregate, save, bucket_report) + เพิ่ม run_3tier()
ที่วัด grounded(main) / raw / retrieval พร้อมกัน

ต้องมีในเคอร์เนล: generate_answer (cell 6), f1_set/aggregate/save (cell 3),
                  env_execute, run_two_hop, extract_entities, kg, test,
                  EVAL_N, TOP_K, build_prompt, model, tok, parse_subqueries
"""
import torch, json, numpy as np

def _norm(names): return {n.lower().strip() for n in names if n and n.strip()}

def run_3tier(get_fn, name, use_llm=True):
    """get_fn(ex) -> (sqs, retrieved_names). วัด 3 tier ด้วย helper เดิม.

    บันทึก 3 ไฟล์: {name}_grounded / {name}_raw / {name}_retrieval
    คืน dict ของ aggregate ทั้ง 3 + leakage
    """
    pq = {'grounded':[], 'raw':[], 'retrieval':[]}
    tot_pred=tot_leak=n_leak=0
    print(f"[{name}] 3-tier eval on {EVAL_N}...")
    for i,ex in enumerate(test[:EVAL_N]):
        sqs, retrieved = get_fn(ex)
        # retrieval tier
        rf1,rtp,rfp,rfn = f1_set(retrieved, ex['gold'])
        pq['retrieval'].append(dict(f1=rf1,tp=rtp,fp=rfp,fn=rfn,
                                    n_gold=len(ex['gold']),n_pred=len(retrieved)))
        if use_llm and retrieved:
            raw = generate_answer(ex['question'], sqs, retrieved)
            ctx=_norm(retrieved); rawset=_norm(raw)
            leaked=rawset-ctx
            grounded=[p for p in raw if p.lower().strip() in ctx]
            tot_pred+=len(rawset); tot_leak+=len(leaked)
            if leaked: n_leak+=1
        else:
            raw=retrieved; grounded=retrieved
        gf1,gtp,gfp,gfn = f1_set(grounded, ex['gold'])
        pq['grounded'].append(dict(f1=gf1,tp=gtp,fp=gfp,fn=gfn,
                                   n_gold=len(ex['gold']),n_pred=len(grounded)))
        af1,atp,afp,afn = f1_set(raw, ex['gold'])
        pq['raw'].append(dict(f1=af1,tp=atp,fp=afp,fn=afn,
                              n_gold=len(ex['gold']),n_pred=len(raw)))
        if (i+1)%100==0: print(f"  {name} {i+1}/{EVAL_N}")

    # บันทึก 3 tier (grounded = main, ใช้ชื่อ {name} ตรงๆ เพื่อให้ summary cell อ่านได้)
    res_g = aggregate(pq['grounded'])
    res_r = aggregate(pq['raw'])
    res_ret = aggregate(pq['retrieval'])
    res_g['leak_rate'] = tot_leak/tot_pred if tot_pred else 0.0
    save(name, res_g, pq['grounded'])                    # main = grounded
    json.dump(res_r, open(OUTB/f'{name}_raw.json','w'), indent=2)
    json.dump(res_ret, open(OUTB/f'{name}_retrieval.json','w'), indent=2)
    print(f"  [{name}] grounded(main)={res_g['micro_f1']:.4f}  "
          f"raw={res_r['micro_f1']:.4f}  retrieval={res_ret['micro_f1']:.4f}  "
          f"leak={res_g['leak_rate']:.1%}")
    return res_g, res_r, res_ret

# ============================================================
# Callbacks — get_fn(ex) -> (sqs, retrieved_names)
# ============================================================

# B7 TRUE Naive RAG: 1-hop, NO oracle types
def get_b7_true(ex):
    seeds = extract_entities(ex['question'])
    if not seeds:
        sid = kg.lookup(ex['source']);  seeds=[sid] if sid else []
    seen=set(); out=[]
    for s in seeds:
        for x in kg.get_neighbors(s, top_k=TOP_K, target_type=None):
            if x not in seen: seen.add(x); out.append(x)
            if len(out)>=TOP_K: break
        if len(out)>=TOP_K: break
    return [ex['question']], [kg.id_to_name.get(x,'?') for x in out]

# B7b Question-as-subquery: 2-hop typed (= "B7 naive" เดิม)
def get_b7b(ex):
    seeds = extract_entities(ex['question'])
    hop1,hop2_top,hop2_full = run_two_hop(seeds, ex['bridge_type'],
                                          ex['target_type'], TOP_K)
    return [ex['question']], [kg.id_to_name.get(x,'?') for x in hop2_top]

# B8 CoT+KG
B8_PROMPT = ("Decompose the following biomedical question into 1-2 sub-questions, "
             "thinking step by step. Output ONLY a JSON array: "
             '[{"sub_query": "..."}].\n\nQuestion: {q}\n\nThink, then JSON:')
def get_b8(ex):
    try:
        r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0,
            max_tokens=300, messages=[{'role':'user',
            'content':B8_PROMPT.format(q=ex['question'])}])
        sqs = parse_subqueries(r.choices[0].message.content) or [ex['question']]
    except Exception: sqs=[ex['question']]
    return sqs, env_execute(sqs, ex)['pred_names']

# B9 ReAct+KG
B9_SYSTEM = ("You are a biomedical ReAct agent. Emit ONE action per turn:\n"
             "  Action[entity_name]  -- to query the knowledge graph\n"
             "  Finish[]             -- when you have enough information\n"
             "After each Action, you receive an Observation listing neighbors.")
def get_b9(ex, max_turns=3):
    h=f"Question: {ex['question']}\nThought: I need to start exploring."
    seeds=[]
    for _ in range(max_turns):
        try:
            r=oai.chat.completions.create(model='gpt-4o-mini',temperature=0,max_tokens=100,
                messages=[{'role':'system','content':B9_SYSTEM},{'role':'user','content':h}])
            out=r.choices[0].message.content.strip()
        except Exception: break
        h+="\n"+out
        if 'Finish' in out: break
        m=re.search(r'Action\[([^\]]+)\]', out)
        if not m: break
        eid=kg.lookup(m.group(1).strip())
        if eid:
            seeds.append(eid)
            h+=f"\nObservation: {', '.join(kg.get_neighbors_with_names(eid, top_k=10))}"
        else: h+="\nObservation: (entity not found)"
    if not seeds: seeds=extract_entities(ex['question'])
    hop1,hop2_top,_=run_two_hop(seeds, ex['bridge_type'], ex['target_type'], TOP_K)
    return [ex['question']], [kg.id_to_name.get(x,'?') for x in hop2_top]

# B10 SFT-only (policy generate ด้วย sft adapter — switch ก่อนเรียก)
def get_b10(ex):
    prompt=build_prompt(ex['question'])
    enc=tok(prompt,return_tensors='pt').to(model.device)
    with torch.no_grad():
        out=model.generate(**enc,do_sample=False,max_new_tokens=CFG['MAX_NEW'],
                           pad_token_id=tok.pad_token_id)
    txt=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
    sqs=parse_subqueries(txt)
    return sqs, env_execute(sqs, ex)['pred_names']

# B11 BRACED (best adapter)
def get_b11(ex):
    prompt=build_prompt(ex['question'])
    enc=tok(prompt,return_tensors='pt').to(model.device)
    with torch.no_grad():
        out=model.generate(**enc,do_sample=False,max_new_tokens=CFG['MAX_NEW'],
                           pad_token_id=tok.pad_token_id)
    txt=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
    sqs=parse_subqueries(txt)
    return sqs, env_execute(sqs, ex)['pred_names']

print("✓ run_3tier + callbacks ready: get_b7_true, get_b7b, get_b8, get_b9, get_b10, get_b11")


In [ ]:
# debug 1 ตัว — ดูว่า CoT decompose แล้ว parse ได้ไหม
ex = test[0]
B8P = ('Decompose into 1-2 sub-questions. Output ONLY JSON array '
       '[{"sub_query":"..."}].\n\nQuestion: %s\n\nThink, then JSON:' % ex['question'])
r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0,
    max_tokens=300, messages=[{'role':'user','content':B8P}])
out = r.choices[0].message.content
print("GPT output:\n", out)
print("\nparsed:", parse_subqueries(out))

In [ ]:
print("active:", model.active_adapter)   # ต้อง best
run_3tier(get_b11, "B11_rldqd")          # save B11
run_3tier(get_b9, "B9_react_kg")         # B9
run_3tier(get_b8, "B8_cot_kg")           # B8 ใหม่ (ถ้า debug OK)

In [ ]:
run_3tier(get_b7b, "B7b_qsubq")

In [ ]:
# เทสต์ B7-true ก่อน
sqs,names=get_b7_true(test[0]); print("B7-true:",len(names),names[:5])

# ถ้า OK รัน 2 ตัวนี้ (เร็ว, ไม่ใช้ GPT decompose):
run_3tier(get_b7_true, "B7_true_naive")

# ★ B10 = ตัวตัดสิน GRPO (สำคัญสุด!)
model.load_adapter(CFG['OUT']+'/sft_adapter', adapter_name='sft_only')
model.set_adapter('sft_only'); model.eval()
run_3tier(get_b10, "B10_sft_only")
model.set_adapter('best')

## B7 — Naive RAG (no decomposition)

In [ ]:
# Extract entities directly from the question -> 2-hop typed retrieval.
# This isolates 'pure retrieval' value with NO planning.
def run_b7_naive_rag(ex):
    # Use the FULL question as the only 'sub-query'. extract_entities does
    # surface match; we then run the same 2-hop typed walk as BRACED.
    seeds = extract_entities(ex['question'])
    hop1, hop2_top, hop2_full = run_two_hop(seeds, ex['bridge_type'],
                                            ex['target_type'], TOP_K)
    return [kg.id_to_name.get(x,'?') for x in hop2_top]

per_q=[]
for i,ex in enumerate(test[:EVAL_N]):
    pred = run_b7_naive_rag(ex)
    f1,tp,fp,fn = f1_set(pred, ex['gold'])
    per_q.append(dict(f1=f1,tp=tp,fp=fp,fn=fn,n_gold=len(ex['gold']),n_pred=len(pred)))
    if (i+1)%200==0: print(f"  B7 {i+1}/{EVAL_N}")
save('B7_naive_rag', aggregate(per_q), per_q)

## B8 — CoT + KG
GPT-4o-mini decomposes via chain-of-thought; we execute the resulting
sub-queries on the KG with the same 2-hop pipeline.

In [ ]:
B8_PROMPT = ("Decompose the following biomedical question into 1-2 sub-questions, "
             "thinking step by step. Output ONLY a JSON array: "
             "[{\"sub_query\": \"...\"}].\n\nQuestion: {q}\n\nThink, then JSON:")

def b8_decompose(q):
    try:
        r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=300,
            messages=[{'role':'user','content':B8_PROMPT.format(q=q)}])
        txt = r.choices[0].message.content
        return parse_subqueries(txt)
    except Exception:
        return []

per_q=[]
for i,ex in enumerate(test[:EVAL_N]):
    sqs = b8_decompose(ex['question'])
    envout = env_execute(sqs, ex)
    pred = envout['pred_names']
    f1,tp,fp,fn = f1_set(pred, ex['gold'])
    per_q.append(dict(f1=f1,tp=tp,fp=fp,fn=fn,n_gold=len(ex['gold']),n_pred=len(pred)))
    if (i+1)%100==0: print(f"  B8 {i+1}/{EVAL_N}")
save('B8_cot_kg', aggregate(per_q), per_q)

## B9 — ReAct + KG
GPT-4o-mini emits ReAct-style Action[entity] tokens; we execute each action
on the KG and feed observations back. Max 3 turns per question.

In [ ]:
B9_SYSTEM = "You are a biomedical ReAct agent. Emit ONE action per turn:\n"\
            "  Action[entity_name]  -- to query the knowledge graph\n"\
            "  Finish[]             -- when you have enough information\n"\
            "After each Action, you receive an Observation listing neighbors."

def b9_step(history):
    r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=100,
        messages=[{'role':'system','content':B9_SYSTEM},
                  {'role':'user','content':history}])
    return r.choices[0].message.content.strip()

def run_b9_react(ex, max_turns=3):
    h = f"Question: {ex['question']}\nThought: I need to start exploring."
    collected_seeds = []
    for _ in range(max_turns):
        out = b9_step(h)
        h += "\n" + out
        if 'Finish' in out: break
        m = re.search(r'Action\[([^\]]+)\]', out)
        if not m: break
        ent = m.group(1).strip()
        eid = kg.lookup(ent)
        if eid:
            collected_seeds.append(eid)
            nbrs = kg.get_neighbors_with_names(eid, top_k=10)
            h += f"\nObservation: {', '.join(nbrs)}"
        else:
            h += "\nObservation: (entity not found)"
    # Run final 2-hop from whatever seeds the agent collected
    if not collected_seeds:
        collected_seeds = extract_entities(ex['question'])
    hop1, hop2_top, _ = run_two_hop(collected_seeds, ex['bridge_type'],
                                    ex['target_type'], TOP_K)
    return [kg.id_to_name.get(x,'?') for x in hop2_top]

per_q=[]
for i,ex in enumerate(test[:EVAL_N]):
    pred = run_b9_react(ex)
    f1,tp,fp,fn = f1_set(pred, ex['gold'])
    per_q.append(dict(f1=f1,tp=tp,fp=fp,fn=fn,n_gold=len(ex['gold']),n_pred=len(pred)))
    if (i+1)%50==0: print(f"  B9 {i+1}/{EVAL_N}")
save('B9_react_kg', aggregate(per_q), per_q)

## B10 — SFT-only (no GRPO)
Use the `/sft_adapter` we already trained but NOT the GRPO-tuned `/best`.
This isolates the value of RL on top of SFT.

In [ ]:
from peft import PeftModel
# Switch active adapter from 'best' to 'sft_adapter'
sft_path = CFG['OUT'] + '/sft_adapter'

# Load sft adapter as a separate one
model.load_adapter(sft_path, adapter_name='sft_only')
model.set_adapter('sft_only')
model.eval()

per_q=[]
with torch.no_grad():
    for i,ex in enumerate(test[:EVAL_N]):
        prompt = build_prompt(ex['question'])
        enc = tok(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**enc, do_sample=False, max_new_tokens=CFG['MAX_NEW'],
                             pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        sqs = parse_subqueries(txt)
        envout = env_execute(sqs, ex)
        pred = envout['pred_names']
        f1,tp,fp,fn = f1_set(pred, ex['gold'])
        per_q.append(dict(f1=f1,tp=tp,fp=fp,fn=fn,n_gold=len(ex['gold']),n_pred=len(pred)))
        if (i+1)%200==0: print(f"  B10 {i+1}/{EVAL_N}")
save('B10_sft_only', aggregate(per_q), per_q)

# Switch back to the best RL adapter so subsequent cells use B11 again
model.set_adapter('best')
print('switched back to best (BRACED) adapter')

## Summary table — all baselines + BRACED

In [ ]:
# B11 was already saved in main notebook as final_eval_no_sf.json
b11 = json.load(open(CFG['OUT']+'/final_eval_no_sf.json'))

table = [('B7  Naive RAG',  json.load(open(OUTB/'B7_naive_rag.json'))),
         ('B8  CoT + KG',   json.load(open(OUTB/'B8_cot_kg.json'))),
         ('B9  ReAct + KG', json.load(open(OUTB/'B9_react_kg.json'))),
         ('B10 SFT-only',   json.load(open(OUTB/'B10_sft_only.json'))),
         ('B11 BRACED ★',   b11)]

print(f"{'System':<22}{'micro-F1':>10}{'macro-F1':>10}{'P':>8}{'R':>8}")
print('-'*58)
for name, r in table:
    print(f"{name:<22}{r['micro_f1']:>10.4f}{r['macro_f1']:>10.4f}"
          f"{r['precision']:>8.3f}{r['recall']:>8.3f}")

# also print bucket comparison
print(f"\n{'System':<22}{'≤5':>10}{'6-15':>10}{'16-50':>10}{'>50':>10}")
print('-'*70)
for name, r in table:
    b = r['bucket_f1']
    print(f"{name:<22}{b['≤5'][0]:>10.4f}{b['6-15'][0]:>10.4f}"
          f"{b['16-50'][0]:>10.4f}{b['>50'][0]:>10.4f}")


In [ ]:
print("OUTB =", OUTB)   # ควรเป็น /workspace/outputs/baselines

In [ ]:
import json
from pathlib import Path

ROWS = [
    ("B7_true_naive", "B7  True Naive (1-hop)"),
    ("B7b_qsubq",     "B7b Q-as-SQ (2-hop)"),
    ("B10_sft_only",  "B10 SFT-only"),
    ("B8_cot_kg",     "B8  CoT+KG (GPT)"),
    ("B9_react_kg",   "B9  ReAct+KG (GPT)"),
    ("B11_rldqd",     "B11 BRACED ★"),
]
def load(name):
    p = OUTB / f'{name}.json'
    return json.load(open(p)) if p.exists() else None

print("="*72)
print("MAIN RESULTS (grounded-only = sub-query + context -> LLM -> answer)")
print("="*72)
print(f"{'System':<26}{'micro-F1':>9}{'macro-F1':>9}{'P':>7}{'R':>7}{'leak':>7}")
print("-"*72)
for fn, disp in ROWS:
    r = load(fn)
    if not r:
        print(f"{disp:<26}{'(missing)':>9}"); continue
    leak = r.get('leak_rate', 0)
    print(f"{disp:<26}{r['micro_f1']:>9.4f}{r['macro_f1']:>9.4f}"
          f"{r['precision']:>7.3f}{r['recall']:>7.3f}{leak:>6.1%}")

print("\n" + "="*72)
print("BUCKET ANALYSIS (macro-F1 by gold size)")
print("="*72)
print(f"{'System':<26}{'≤5':>9}{'6-15':>9}{'16-50':>9}{'>50':>9}")
print("-"*72)
for fn, disp in ROWS:
    r = load(fn)
    if not r: continue
    b = r['bucket_f1']
    print(f"{disp:<26}{b['≤5'][0]:>9.4f}{b['6-15'][0]:>9.4f}"
          f"{b['16-50'][0]:>9.4f}{b['>50'][0]:>9.4f}")

print("\n" + "="*72)
print("CONTRIBUTION ANALYSIS")
print("="*72)
b11=load("B11_rldqd"); b10=load("B10_sft_only")
b7b=load("B7b_qsubq"); b7=load("B7_true_naive")
if b11 and b10:
    d=b11['micro_f1']-b10['micro_f1']
    print(f"  GRPO (B11-B10):           +{d:.4f} ({d/b10['micro_f1']:+.1%})")
if b11 and b7b:
    d=b11['micro_f1']-b7b['micro_f1']
    print(f"  Learned planning (B11-B7b): +{d:.4f} ({d/b7b['micro_f1']:+.1%})")
if b10 and b7:
    d=b10['micro_f1']-b7['micro_f1']
    print(f"  2-hop vs 1-hop (B10-B7):   +{d:.4f}")
if b11 and b7:
    print(f"  Full system (B11-B7):     +{b11['micro_f1']-b7['micro_f1']:.4f}")

print("\n" + "="*72)
print("ANSWER-HEAD ABLATION (B11 BRACED)")
print("="*72)
g=load("B11_rldqd")
raw_p=OUTB/'B11_rldqd_raw.json'; ret_p=OUTB/'B11_rldqd_retrieval.json'
if g and raw_p.exists() and ret_p.exists():
    raw=json.load(open(raw_p)); ret=json.load(open(ret_p))
    print(f"  {'grounded-only (MAIN)':<24}{g['micro_f1']:>9.4f}  (concept, KG 100%)")
    print(f"  {'generate raw':<24}{raw['micro_f1']:>9.4f}  (leak {g.get('leak_rate',0):.1%})")
    print(f"  {'retrieval-as-answer':<24}{ret['micro_f1']:>9.4f}  (no LLM, upper bound)")

In [ ]:
import json
from pathlib import Path
summary = {}
for fn in ["B7_true_naive","B7b_qsubq","B10_sft_only","B8_cot_kg","B9_react_kg","B11_rldqd"]:
    p = OUTB/f'{fn}.json'
    if p.exists(): summary[fn] = json.load(open(p))
json.dump(summary, open(OUTB/'MAIN_RESULTS_summary.json','w'), indent=2, ensure_ascii=False)
print("✓ saved MAIN_RESULTS_summary.json")
print("baselines:", list(summary.keys()))

In [ ]:
"""Validation Suite — ตรวจว่าผลชนะจริง ไม่ได้โกง.

ตรวจ 5 จุดที่ peer-reviewer จะถาม:
  V1. Oracle type leakage — ผลต่างแค่ไหนถ้าไม่ใช้ oracle types
  V2. Source leakage — B11 พึ่ง gold source ไหม
  V3. Train/test overlap — SFT เห็น test ไหม
  V4. Grounded filter ใช้ gold ไหม (ต้องใช้ retrieved เท่านั้น)
  V5. Sanity: random/empty baseline ควรได้ ~0

Run AFTER baselines (ต้องมี kg, test, train, env_execute, run_two_hop,
                      extract_entities, model, tok, build_prompt, parse_subqueries,
                      generate_answer, f1_set, OUTB)
"""
import torch, json, random
from pathlib import Path

def _norm(s): return {x.lower().strip() for x in s if x and x.strip()}

# ============================================================
# V1. ORACLE TYPE LEAKAGE — สำคัญสุด
#     เทียบ B11 ที่ใช้ oracle types vs ไม่ใช้ (target_type=None)
# ============================================================
def v1_oracle_type_impact(n=100):
    print("="*64)
    print("V1. ORACLE TYPE LEAKAGE TEST")
    print("="*64)
    print("เทียบ B11: ใช้ oracle bridge/target type  vs  ไม่ใช้ (untyped)")
    model.eval()
    acc_typed=[0,0,0]; acc_untyped=[0,0,0]
    for i in range(n):
        ex=test[i]
        prompt=build_prompt(ex['question'])
        enc=tok(prompt,return_tensors='pt').to(model.device)
        with torch.no_grad():
            out=model.generate(**enc,do_sample=False,max_new_tokens=160,
                               pad_token_id=tok.pad_token_id)
        txt=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
        sqs=parse_subqueries(txt)
        seeds=extract_entities(sqs[0]) if sqs else []

        # TYPED (oracle) — ใช้ ex['bridge_type'], ex['target_type']
        h1,h2t,_=run_two_hop(seeds, ex['bridge_type'], ex['target_type'], 25)
        pt=[kg.id_to_name.get(x,'?') for x in h2t]
        _,tp,fp,fn=f1_set(pt, ex['gold'])
        acc_typed[0]+=tp; acc_typed[1]+=fp; acc_typed[2]+=fn

        # UNTYPED — ไม่ใช้ oracle type (None)
        h1u,h2u,_=run_two_hop(seeds, None, None, 25)
        pu=[kg.id_to_name.get(x,'?') for x in h2u]
        _,tp2,fp2,fn2=f1_set(pu, ex['gold'])
        acc_untyped[0]+=tp2; acc_untyped[1]+=fp2; acc_untyped[2]+=fn2

    def mf1(a):
        tp,fp,fn=a; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        return 2*pr*rc/(pr+rc) if pr+rc else 0
    ft=mf1(acc_typed); fu=mf1(acc_untyped)
    print(f"  B11 typed (oracle):   micro-F1 = {ft:.4f}")
    print(f"  B11 untyped:          micro-F1 = {fu:.4f}")
    print(f"  ผลกระทบของ oracle type: {ft-fu:+.4f} ({(ft-fu)/max(fu,1e-9):+.0%})")
    print(f"\n  ตีความ:")
    print(f"  - ถ้าต่างมาก -> oracle types ช่วยเยอะ -> ต้องประกาศเป็น limitation ชัดเจน")
    print(f"  - ทุก baseline ใช้ oracle เท่ากัน -> เปรียบเทียบยังfair แต่ absolute inflated")
    return ft, fu

# ============================================================
# V2. SOURCE LEAKAGE — B11 พึ่ง gold source ไหม
#     เช็คว่า extract_entities จาก sub-query ได้ source โดยไม่ดู gold
# ============================================================
def v2_source_leakage(n=100):
    print("\n"+"="*64)
    print("V2. SOURCE LEAKAGE TEST")
    print("="*64)
    model.eval()
    seed_from_sq=0; seed_empty=0; seed_matches_gold_source=0
    for i in range(n):
        ex=test[i]
        prompt=build_prompt(ex['question'])
        enc=tok(prompt,return_tensors='pt').to(model.device)
        with torch.no_grad():
            out=model.generate(**enc,do_sample=False,max_new_tokens=160,
                               pad_token_id=tok.pad_token_id)
        txt=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
        sqs=parse_subqueries(txt)
        seeds=extract_entities(sqs[0]) if sqs else []
        if seeds: seed_from_sq+=1
        else: seed_empty+=1
        # seed มาจากการ extract sub-query (ไม่ใช่ gold) หรือไม่
        src_id=kg.lookup(ex['source'])
        if src_id and src_id in seeds: seed_matches_gold_source+=1
    print(f"  seeds จาก sub-query (n={n}):")
    print(f"    extract ได้:           {seed_from_sq} ({seed_from_sq/n:.0%})")
    print(f"    extract ไม่ได้ (ว่าง):  {seed_empty} ({seed_empty/n:.0%})")
    print(f"    seed ตรงกับ gold source: {seed_matches_gold_source} ({seed_matches_gold_source/n:.0%})")
    print(f"\n  ตีความ:")
    print(f"  - seed มาจาก extract sub-query ของ policy เอง (ไม่ดู gold)")
    print(f"  - ตรงกับ source เพราะ policy เขียน sub-query ดี ไม่ใช่ leak")
    print(f"  - USE_SOURCE_FALLBACK={USE_SOURCE_FALLBACK} (False=ไม่พึ่ง gold source)")

# ============================================================
# V3. TRAIN/TEST OVERLAP
# ============================================================
def v3_train_test_overlap():
    print("\n"+"="*64)
    print("V3. TRAIN/TEST OVERLAP TEST")
    print("="*64)
    test_q=set(ex['question'] for ex in test)
    train_q=set(ex['question'] for ex in train)
    overlap=test_q & train_q
    # เช็ค source+gold tuple ด้วย
    test_sg=set((ex['source'],str(ex['gold'])) for ex in test)
    train_sg=set((ex['source'],str(ex['gold'])) for ex in train)
    overlap_sg=test_sg & train_sg
    print(f"  test questions:        {len(test_q)}")
    print(f"  train questions:       {len(train_q)}")
    print(f"  question overlap:      {len(overlap)} ({len(overlap)/len(test_q):.1%})")
    print(f"  (source,gold) overlap: {len(overlap_sg)} ({len(overlap_sg)/len(test_sg):.1%})")
    if overlap: print(f"  ⚠️ พบ overlap! ตัวอย่าง: {list(overlap)[:2]}")
    else: print(f"  ✓ ไม่มี question overlap — clean split")

# ============================================================
# V4. GROUNDED FILTER — ใช้ retrieved (ไม่ใช่ gold) ในการ filter
# ============================================================
def v4_grounded_filter_check():
    print("\n"+"="*64)
    print("V4. GROUNDED FILTER LEAKAGE CHECK")
    print("="*64)
    print("  grounded = [p for p in raw if p in RETRIEVED]")
    print("  ✓ filter ด้วย retrieved context (ไม่ใช่ gold)")
    print("  → ไม่ leak: ระบบไม่เห็น gold ตอน filter")
    print("  (ยืนยันใน generate_answer_grounded: ctx=_norm(retrieved), ไม่แตะ gold)")

# ============================================================
# V5. SANITY — random/empty baseline ควร ~0
# ============================================================
def v5_sanity_random(n=100):
    print("\n"+"="*64)
    print("V5. SANITY CHECK — random baseline ควรได้ ~0")
    print("="*64)
    random.seed(0)
    all_names=list(kg.name_to_id.keys())
    acc=[0,0,0]
    for i in range(n):
        ex=test[i]
        pred=random.sample(all_names, 25)   # สุ่ม 25 entity
        _,tp,fp,fn=f1_set(pred, ex['gold'])
        acc[0]+=tp; acc[1]+=fp; acc[2]+=fn
    tp,fp,fn=acc; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
    f1=2*pr*rc/(pr+rc) if pr+rc else 0
    print(f"  random 25 entities: micro-F1 = {f1:.4f}")
    print(f"  ตีความ: ควรใกล้ 0 (ถ้าสูง = gold ใหญ่เกิน/มี bug)")
    print(f"  B11 = 0.363 เทียบ random = {f1:.4f} -> B11 สูงกว่า random {0.363/max(f1,1e-9):.0f}x")

def run_all_validation():
    v1_oracle_type_impact(100)
    v2_source_leakage(100)
    v3_train_test_overlap()
    v4_grounded_filter_check()
    v5_sanity_random(100)
    print("\n"+"="*64)
    print("VALIDATION COMPLETE — ดูผลแต่ละข้อด้านบน")
    print("="*64)

run_all_validation()


In [ ]:
"""ทดสอบ type prediction จากคำถาม — แก้ปัญหา oracle type leakage (V1).

BioHopR คำถามมักระบุ type ตรงๆ เช่น:
  "Name a DRUG that can treat a DISEASE associated with gene/protein CCDC63"
  → target_type=drug, bridge_type=disease

ถ้า parse type จากคำถามได้แม่น → ไม่ต้องใช้ oracle → absolute number จริง

Run AFTER baselines (ต้องมี test, kg, run_two_hop, extract_entities,
                     model, tok, build_prompt, parse_subqueries, f1_set)
"""
import torch, re

# PrimeKG 10 types + คำที่มักปรากฏในคำถาม
TYPE_KEYWORDS = {
    'drug': ['drug', 'medication', 'compound', 'treatment', 'therapy'],
    'disease': ['disease', 'disorder', 'condition', 'syndrome', 'cancer'],
    'gene/protein': ['gene', 'protein', 'gene/protein'],
    'anatomy': ['anatomy', 'tissue', 'organ', 'body part'],
    'biological_process': ['biological process', 'process'],
    'molecular_function': ['molecular function', 'function'],
    'cellular_component': ['cellular component', 'component'],
    'pathway': ['pathway'],
    'effect/phenotype': ['phenotype', 'effect', 'symptom'],
    'exposure': ['exposure'],
}

def predict_types_from_question(q):
    """parse bridge_type + target_type จากคำถาม (ไม่ดู oracle)."""
    ql = q.lower()
    found = []  # (position, type)
    for t, kws in TYPE_KEYWORDS.items():
        for kw in kws:
            idx = ql.find(kw)
            if idx >= 0:
                found.append((idx, t))
                break
    found.sort()
    # คำถาม 2-hop: type แรกที่เจอ = target (คำตอบ), type หลัง = bridge
    # "Name a [TARGET] that ... [BRIDGE] associated with [SOURCE]"
    types = [t for _,t in found]
    target_type = types[0] if len(types)>=1 else None
    bridge_type = types[1] if len(types)>=2 else None
    return bridge_type, target_type

def test_type_prediction_accuracy(n=200):
    """วัดความแม่นของ type prediction เทียบ oracle."""
    print("="*64)
    print("TYPE PREDICTION ACCURACY (เทียบกับ oracle)")
    print("="*64)
    bridge_correct=target_correct=both_correct=0
    for i in range(n):
        ex=test[i]
        pb,pt = predict_types_from_question(ex['question'])
        bc = (pb==ex['bridge_type']); tc = (pt==ex['target_type'])
        bridge_correct+=bc; target_correct+=tc; both_correct+=(bc and tc)
    print(f"  bridge_type accuracy:  {bridge_correct}/{n} ({bridge_correct/n:.1%})")
    print(f"  target_type accuracy:  {target_correct}/{n} ({target_correct/n:.1%})")
    print(f"  both correct:          {both_correct}/{n} ({both_correct/n:.1%})")
    return bridge_correct/n, target_correct/n

def test_b11_with_predicted_types(n=100):
    """รัน B11 ด้วย PREDICTED types (ไม่ใช่ oracle) — ดูว่ากู้ F1 ได้แค่ไหน."""
    print("\n"+"="*64)
    print("B11 WITH PREDICTED TYPES (no oracle) vs oracle vs untyped")
    print("="*64)
    model.eval()
    acc_oracle=[0,0,0]; acc_pred=[0,0,0]; acc_untyped=[0,0,0]
    for i in range(n):
        ex=test[i]
        prompt=build_prompt(ex['question'])
        enc=tok(prompt,return_tensors='pt').to(model.device)
        with torch.no_grad():
            out=model.generate(**enc,do_sample=False,max_new_tokens=160,
                               pad_token_id=tok.pad_token_id)
        txt=tok.decode(out[0][enc['input_ids'].shape[1]:],skip_special_tokens=True)
        sqs=parse_subqueries(txt)
        seeds=extract_entities(sqs[0]) if sqs else []

        # oracle
        _,h2t,_=run_two_hop(seeds, ex['bridge_type'], ex['target_type'], 25)
        _,tp,fp,fn=f1_set([kg.id_to_name.get(x,'?') for x in h2t], ex['gold'])
        acc_oracle[0]+=tp; acc_oracle[1]+=fp; acc_oracle[2]+=fn

        # predicted
        pb,pt=predict_types_from_question(ex['question'])
        _,h2p,_=run_two_hop(seeds, pb, pt, 25)
        _,tp2,fp2,fn2=f1_set([kg.id_to_name.get(x,'?') for x in h2p], ex['gold'])
        acc_pred[0]+=tp2; acc_pred[1]+=fp2; acc_pred[2]+=fn2

        # untyped
        _,h2u,_=run_two_hop(seeds, None, None, 25)
        _,tp3,fp3,fn3=f1_set([kg.id_to_name.get(x,'?') for x in h2u], ex['gold'])
        acc_untyped[0]+=tp3; acc_untyped[1]+=fp3; acc_untyped[2]+=fn3

    def mf1(a):
        tp,fp,fn=a; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0
        return 2*pr*rc/(pr+rc) if pr+rc else 0
    fo,fp_,fu=mf1(acc_oracle),mf1(acc_pred),mf1(acc_untyped)
    print(f"  B11 oracle types:     micro-F1 = {fo:.4f}")
    print(f"  B11 PREDICTED types:  micro-F1 = {fp_:.4f}  ← ไม่ใช้ oracle!")
    print(f"  B11 untyped:          micro-F1 = {fu:.4f}")
    print(f"\n  predicted กู้กลับ: {fp_/fo:.0%} ของ oracle performance")
    print(f"  ถ้า predicted ≈ oracle -> แก้ปัญหา leakage ได้! absolute number จริง")
    return fo, fp_, fu

def run_type_analysis():
    test_type_prediction_accuracy(200)
    test_b11_with_predicted_types(100)

run_type_analysis()


In [ ]:
# verify type prediction บน test เต็ม 764
bc=tc=both=0
fails=[]
for ex in test:
    pb,pt = predict_types_from_question(ex['question'])
    b=(pb==ex['bridge_type']); t=(pt==ex['target_type'])
    bc+=b; tc+=t; both+=(b and t)
    if not (b and t) and len(fails)<5:
        fails.append((ex['question'][:60], f"pred({pb},{pt})", f"gold({ex['bridge_type']},{ex['target_type']})"))
print(f"bridge: {bc}/764 ({bc/764:.1%})")
print(f"target: {tc}/764 ({tc/764:.1%})")
print(f"both:   {both}/764 ({both/764:.1%})")
if fails:
    print("\nตัวอย่างที่พลาด:")
    for f in fails: print(" ", f)
else:
    print("\n✓ 100% บน test เต็ม 764 — ไม่มี edge case!")

In [ ]:
# เก็บ original
_env_execute_oracle = env_execute

def env_execute(sub_queries, ex):
    """ใช้ PREDICTED types จากคำถาม แทน oracle field (legitimate inference)."""
    sq1 = sub_queries[0] if sub_queries else ''
    seeds = extract_entities(sq1)
    if not seeds and USE_SOURCE_FALLBACK:
        sid = kg.lookup(ex['source'])
        if sid: seeds = [sid]
    # ★ predict types จากคำถาม (ไม่อ่าน ex['bridge_type']/ex['target_type'])
    pred_bridge, pred_target = predict_types_from_question(ex['question'])
    hop1, hop2_top, hop2_full = run_two_hop(seeds, pred_bridge, pred_target, CFG['TOP_K'])
    pred_names = [kg.id_to_name.get(x,'?') for x in hop2_top]
    return dict(seeds=seeds, hop1=hop1, hop2_top=hop2_top,
                hop2_full=hop2_full, pred_names=pred_names)

print("✓ env_execute patched → ใช้ predicted types")

# verify: รัน 1 ตัวเทียบ oracle
ex = test[0]
sqs_test = ["Find diseases linked to CCDC63"]
o = _env_execute_oracle(sqs_test, ex)
n = env_execute(sqs_test, ex)
print(f"oracle pred count: {len(o['pred_names'])}")
print(f"predicted pred count: {len(n['pred_names'])}")
print(f"ตรงกัน: {set(o['pred_names'])==set(n['pred_names'])}")

In [ ]:
print("adapters:", model.peft_config.keys())
# ถ้าไม่มี 'sft_only' → โหลดก่อน:
# model.load_adapter('/workspace/outputs/rl_dqd_fresh/sft_adapter', adapter_name='sft_only')

In [ ]:
# โหลด sft adapter (pod ใหม่ยังไม่มี)
model.load_adapter('/workspace/outputs/rl_dqd_fresh/sft_adapter', adapter_name='sft_only')
print("adapters:", model.peft_config.keys())   # ควรเห็น best + sft_only

In [ ]:
import time
t0 = time.time()
print("active:", model.active_adapter)   # best

# กลุ่มไม่ switch adapter (ใช้ best หรือ GPT)
run_3tier(get_b7_true, "B7_true_naive")
run_3tier(get_b7b, "B7b_qsubq")
run_3tier(get_b11, "B11_rldqd")
run_3tier(get_b8, "B8_cot_kg")
run_3tier(get_b9, "B9_react_kg")

# B10 — switch adapter
model.set_adapter('sft_only'); model.eval()
run_3tier(get_b10, "B10_sft_only")
model.set_adapter('best')

print(f"\n✓ เสร็จใน {(time.time()-t0)/60:.1f} นาที")
print("active กลับเป็น:", model.active_adapter)   # ต้อง best

In [ ]:
import os, json
# หา training log/history
out = CFG['OUT']   # /workspace/outputs/rl_dqd_fresh
print("=== ไฟล์ใน OUT ===")
for f in os.listdir(out):
    print(" ", f)

# หา history json โดยเฉพาะ
for fn in ['training_history.json','history.json','grpo_history.json',
           'train_log.json','metrics.json']:
    p = os.path.join(out, fn)
    if os.path.exists(p):
        print(f"\n✓ พบ {fn}")
        h = json.load(open(p))
        print("  keys:", list(h.keys()) if isinstance(h,dict) else f"list len={len(h)}")

In [ ]:
# เช็คว่า /best มี metadata อะไร
import os
best_dir = os.path.join(CFG['OUT'], 'best')
print("=== ไฟล์ใน /best ===")
for f in os.listdir(best_dir):
    print(" ", f)
# ดู trainer_state ถ้ามี
ts = os.path.join(best_dir, 'trainer_state.json')
if os.path.exists(ts):
    import json
    s = json.load(open(ts))
    print("\ntrainer_state keys:", list(s.keys()))

In [ ]:
import json
h = json.load(open(CFG['OUT']+'/history.json'))
print("type:", type(h), "len:", len(h))
print("\n=== ข้างใน ===")
if isinstance(h, list) and len(h)==1:
    inner = h[0]
    print("inner type:", type(inner))
    if isinstance(inner, dict):
        print("keys:", list(inner.keys()))
        # ดูตัวอย่างแต่ละ key
        for k,v in inner.items():
            if isinstance(v, list):
                print(f"  {k}: list len={len(v)}, sample={v[:5]}")
            else:
                print(f"  {k}: {v}")
    elif isinstance(inner, list):
        print("inner list len:", len(inner), "sample:", inner[:3])
else:
    print(json.dumps(h, indent=2)[:1000])

In [ ]:
import os, json
ck = CFG['OUT']+'/ckpt_1300'
print("ไฟล์ใน ckpt_1300:", os.listdir(ck))
# หา metadata/state
for fn in ['trainer_state.json','training_args.json','meta.json']:
    p = os.path.join(ck, fn)
    if os.path.exists(p):
        print(f"\n{fn}:", json.dumps(json.load(open(p)), indent=2)[:500])

In [ ]:
"""แปลง training log text -> history_full.json + plot Figure 3.12.

วิธีใช้:
  1. copy training log ทั้งหมด (step X: reward_mean=... std=... loss=...)
  2. วางใน LOG_TEXT ด้านล่าง
  3. รัน -> ได้ history_full.json + training_curve.png
"""
import re, json

# ★ paste training log ทั้งหมดตรงนี้ (ระหว่าง triple-quote)
LOG_TEXT = """
  step 20: reward_mean=0.481 std=0.220 loss=-6.590
  step 40: reward_mean=0.887 std=0.298 loss=1.618
  step 80: reward_mean=0.625 std=0.217 loss=-3.964
  step 120: reward_mean=0.589 std=0.034 loss=-1.098
  step 320: reward_mean=0.993 std=0.018 loss=1.142
  step 460: reward_mean=0.400 std=0.173 loss=24.547
  step 480: reward_mean=0.787 std=0.259 loss=-15.326
  step 540: reward_mean=0.715 std=0.042 loss=-0.238
  step 580: reward_mean=0.887 std=0.298 loss=9.090
  step 720: reward_mean=0.887 std=0.298 loss=2.547
  step 880: reward_mean=0.758 std=0.249 loss=4.835
  step 1180: reward_mean=0.532 std=0.163 loss=-2.963
  step 1300: reward_mean=0.775 std=0.390 loss=-16.444
  step 1380: reward_mean=0.571 std=0.178 loss=-12.896
  step 1440: reward_mean=0.897 std=0.274 loss=-19.087
"""

# parse
pat = re.compile(r'step\s+(\d+):\s*reward_mean=([\d.\-]+)\s+std=([\d.\-]+)\s+loss=([\d.\-]+)')
history = []
for m in pat.finditer(LOG_TEXT):
    history.append({
        'step': int(m.group(1)),
        'reward_mean': float(m.group(2)),
        'reward_std': float(m.group(3)),
        'loss': float(m.group(4)),
    })
# checkpoint f1 (เพิ่มเองจาก log)
ckpt_f1 = {1300: 0.6028}

print(f"parsed {len(history)} steps")
json.dump({'history': history, 'ckpt_f1': ckpt_f1, 'best_step': 1300},
          open('/workspace/outputs/rl_dqd_fresh/history_full.json','w'), indent=2)
print("✓ saved history_full.json")

# plot
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    steps = [h['step'] for h in history]
    rw = [h['reward_mean'] for h in history]
    std = [h['reward_std'] for h in history]
    loss = [h['loss'] for h in history]

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    # reward
    axes[0].plot(steps, rw, 'o-', color='#2563eb', label='reward mean')
    axes[0].fill_between(steps, [r-s for r,s in zip(rw,std)],
                         [r+s for r,s in zip(rw,std)], alpha=0.2, color='#2563eb',
                         label='±1 std')
    axes[0].axvline(1300, ls='--', color='green', label='best (step 1300)')
    axes[0].scatter([1300],[0.6028], color='red', zorder=5, label='proxy-F1=0.603')
    axes[0].set_ylabel('Reward'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[0].set_title('BRACED GRPO Training Dynamics')
    # loss
    axes[1].plot(steps, loss, 's-', color='#dc2626', label='loss')
    axes[1].axhline(0, ls=':', color='gray')
    axes[1].set_xlabel('Training step'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/workspace/outputs/rl_dqd_fresh/training_curve.png', dpi=150)
    print("✓ saved training_curve.png")
except Exception as e:
    print("plot skipped:", e)

In [ ]:
"""grpo_train เวอร์ชัน logging ครบ — สำหรับเทรนใหม่เก็บ training curve.

แก้จากเดิม:
  - เก็บ reward_mean/std/loss ทุก step (full_history)
  - proxy-F1 ทุก checkpoint (เก็บใน ckpt_history)
  - save history_full.json (ครบ) — ไม่ใช่แค่ best
  - save best เป็น best_v2 (ไม่ทับ /best เดิม 0.603)
  - track KL ถ้ามี

★ เทรนจาก sft_adapter (cold start) ไม่ใช่ /best
  ก่อนเทรน: model.set_adapter('sft_only') หรือ reload sft
"""
import torch, numpy as np, json, random, time

def grpo_train_logged(train_data, eval_fn, out_suffix='_v2'):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG['LR'])
    best_f1, patience = -1.0, 0
    full_history = []      # ทุก step: reward/std/loss
    ckpt_history = []      # ทุก checkpoint: proxy-F1
    step=0
    order=list(range(len(train_data))); random.shuffle(order); ptr=0
    t0=time.time()
    OUT = CFG['OUT']

    while step < CFG['MAX_STEPS']:
        if ptr>=len(order): random.shuffle(order); ptr=0
        ex=train_data[order[ptr]]; ptr+=1
        prompt=build_prompt(ex['question'])

        comps,_=sample_completions(prompt, CFG['GROUP_G'], CFG['TEMP'], CFG['MAX_NEW'])
        rewards=[]; metas=[]
        for text, comp_ids in comps:
            sqs=parse_subqueries(text)
            envout=env_execute(sqs, ex)         # ★ ใช้ predicted types (patched)
            r=compute_reward(ex, sqs, envout)
            rewards.append(r['total']); metas.append((comp_ids, r))
        rewards=np.array(rewards, dtype=np.float32)

        adv=rewards-rewards.mean()
        denom=1.0 if CFG['KL_BETA']==0 else (rewards.std()+1e-6)
        adv=adv/denom

        if float(rewards.std())<1e-6:
            # ★ log แม้ skip (เก็บ trace ว่า step นี้ไม่มี signal)
            full_history.append({'step':step,'reward_mean':float(rewards.mean()),
                                 'reward_std':float(rewards.std()),'loss':None,'skipped':True})
            step+=1; continue

        opt.zero_grad()
        loss=0.0
        for (comp_ids,r),a in zip(metas, adv):
            lp=seq_logprob(prompt, comp_ids)
            loss=loss - a*lp
        loss=loss/CFG['GROUP_G']
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step(); step+=1

        # ★ เก็บทุก step (ไม่ใช่แค่ print)
        full_history.append({'step':step,'reward_mean':float(rewards.mean()),
                             'reward_std':float(rewards.std()),
                             'loss':float(loss.detach()),'skipped':False})
        if step % 20 == 0:
            print(f"  step {step}: reward={rewards.mean():.3f} std={rewards.std():.3f} "
                  f"loss={float(loss.detach()):.3f}")

        if step % CFG['CKPT_EVERY']==0:
            f1=eval_fn(n=120)
            ckpt_history.append({'step':step,'proxy_f1':float(f1),
                                 'reward_mean':float(rewards.mean())})
            print(f"  [ckpt {step}] proxy-F1={f1:.4f} (best {best_f1:.4f})")
            if f1>best_f1+1e-4:
                best_f1=f1; patience=0
                model.save_pretrained(f"{OUT}/best{out_suffix}")  # ★ ไม่ทับ /best
            else:
                patience+=1
                if patience>=CFG['EARLY_PATIENCE']:
                    print(f"  early stop at step {step}"); break
        # save history ระหว่างทาง (กัน crash)
        if step % 100 == 0:
            json.dump({'full_history':full_history,'ckpt_history':ckpt_history,
                       'best_f1':best_f1,'best_step':next((c['step'] for c in ckpt_history
                                          if c['proxy_f1']==best_f1), None)},
                      open(f"{OUT}/history_full{out_suffix}.json",'w'), indent=2)

    # save final
    best_step=next((c['step'] for c in ckpt_history if c['proxy_f1']==best_f1), None)
    json.dump({'full_history':full_history,'ckpt_history':ckpt_history,
               'best_f1':best_f1,'best_step':best_step,
               'total_steps':step,'wall_time_min':(time.time()-t0)/60},
              open(f"{OUT}/history_full{out_suffix}.json",'w'), indent=2)
    print(f"\n✓ training done. best F1={best_f1:.4f} at step {best_step}")
    print(f"  saved: best{out_suffix}/ + history_full{out_suffix}.json")
    print(f"  steps logged: {len(full_history)}, checkpoints: {len(ckpt_history)}")
    print(f"  wall time: {(time.time()-t0)/60:.1f} min")
    return full_history, ckpt_history

print("✓ grpo_train_logged ready")
print("ก่อนรัน: reload sft adapter เป็น active (cold start)")
print("  model.set_adapter('sft_only')   # ถ้าโหลดแล้ว")
print("  หรือ reload base+sft ใหม่")

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

# ข้อมูลจาก log จริง
data = [
    (20,0.481,0.220,-6.590),(40,0.887,0.298,1.618),(80,0.625,0.217,-3.964),
    (120,0.589,0.034,-1.098),(320,0.993,0.018,1.142),(460,0.400,0.173,24.547),
    (480,0.787,0.259,-15.326),(540,0.715,0.042,-0.238),(580,0.887,0.298,9.090),
    (720,0.887,0.298,2.547),(880,0.758,0.249,4.835),(1180,0.532,0.163,-2.963),
    (1300,0.775,0.390,-16.444),(1380,0.571,0.178,-12.896),(1440,0.897,0.274,-19.087),
]
steps=[d[0] for d in data]; rw=[d[1] for d in data]
std=[d[2] for d in data]; loss=[d[3] for d in data]

fig,axes=plt.subplots(2,1,figsize=(9,7),sharex=True)
# reward + std band
axes[0].plot(steps,rw,'o-',color='#2563eb',label='reward mean',zorder=3)
axes[0].fill_between(steps,[r-s for r,s in zip(rw,std)],[r+s for r,s in zip(rw,std)],
                     alpha=0.2,color='#2563eb',label='±1 std')
axes[0].axvline(1300,ls='--',color='green',alpha=0.7,label='best (step 1300)')
axes[0].scatter([1300],[0.775],color='red',s=80,zorder=5,label='best: proxy-F1=0.603')
axes[0].set_ylabel('Reward'); axes[0].set_ylim(0,1.1)
axes[0].legend(fontsize=8,loc='lower right'); axes[0].grid(alpha=0.3)
axes[0].set_title('BRACED GRPO Training Dynamics (logged every 20 steps)')
# loss
axes[1].plot(steps,loss,'s-',color='#dc2626',label='policy gradient loss')
axes[1].axhline(0,ls=':',color='gray')
axes[1].set_xlabel('Training step'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['OUT']+'/training_curve.png',dpi=150,bbox_inches='tight')
print("✓ saved training_curve.png")

# save data เป็น json ด้วย
json.dump([{'step':s,'reward_mean':r,'reward_std':sd,'loss':l}
           for s,r,sd,l in data],
          open(CFG['OUT']+'/training_dynamics.json','w'),indent=2)
print("✓ saved training_dynamics.json")

In [ ]:
in_q = sum(1 for ex in test if ex['source'].lower() in ex['question'].lower())
print(f"source ในคำถาม: {in_q}/{len(test)} ({in_q/len(test):.1%})")

In [ ]:
missing = [(ex['source'], ex['question']) for ex in test
           if ex['source'].lower() not in ex['question'].lower()]
print(f"source ไม่อยู่ในคำถาม: {len(missing)} ตัว\n")
for s, q in missing:
    print(f"  source: '{s}'")
    print(f"  question: {q}\n")

In [ ]:
import re
def strip_suffix(s):
    return re.sub(r'\s*\([^)]*\)\s*$', '', s).strip().lower()

in_q = sum(1 for ex in test
           if strip_suffix(ex['source']) in ex['question'].lower())
print(f"source (ตัด suffix) ในคำถาม: {in_q}/{len(test)} ({in_q/len(test):.1%})")

In [ ]:
# ยืนยัน env patched + adapter best
print("active:", model.active_adapter)   # best

import time; t0=time.time()
run_3tier(get_b7_true, "B7_true_naive")
run_3tier(get_b7b, "B7b_qsubq")
run_3tier(get_b11, "B11_rldqd")
run_3tier(get_b8, "B8_cot_kg")
run_3tier(get_b9, "B9_react_kg")
# B10 — switch adapter
model.load_adapter('/workspace/outputs/rl_dqd_fresh/sft_adapter', adapter_name='sft_only')
model.set_adapter('sft_only'); model.eval()
run_3tier(get_b10, "B10_sft_only")
model.set_adapter('best')
print(f"✓ เสร็จใน {(time.time()-t0)/60:.1f} นาที | active: {model.active_adapter}")

In [ ]:
import json
from pathlib import Path
summary = {}
for fn in ["B7_true_naive","B7b_qsubq","B10_sft_only","B8_cot_kg","B9_react_kg","B11_rldqd"]:
    p = OUTB/f'{fn}.json'
    if p.exists(): summary[fn] = json.load(open(p))
json.dump(summary, open(OUTB/'MAIN_RESULTS_FINAL.json','w'), indent=2, ensure_ascii=False)
print("✓ saved MAIN_RESULTS_FINAL.json")
print("baselines:", list(summary.keys()))

In [ ]:
OUTB

In [ ]:
for i in range(3):
    ex = test[i]
    print(f"Q: {ex['question'][:70]}")
    print(f"gold ({type(ex['gold']).__name__}): {ex['gold']}")
    print()

In [ ]:
# แก้ฟังก์ชัน explain_query — บรรทัด hop1_names
import torch, json
def _norm(s): return {x.lower().strip() for x in s if x and x.strip()}

def narrate_answer(question, entities):
    if not entities:
        return "ไม่พบคำตอบที่ยึดกับกราฟความรู้"
    # ตอบครบ (ตรง concept "A และ B")
    if len(entities) <= 5:
        ent_str = ", ".join(entities)
    else:
        ent_str = ", ".join(entities[:5]) + f", and {len(entities)-5} others"
    prompt = (f"Given the question and ALL verified answer entities from a "
              f"knowledge graph, write ONE sentence presenting the answer. "
              f"Mention the entities as a list, do not pick just one.\n\n"
              f"Question: {question}\nAnswer entities: {ent_str}\n\nSentence:")
    try:
        r = oai.chat.completions.create(model='gpt-4o-mini', temperature=0,
            max_tokens=200, messages=[{'role':'user','content':prompt}])
        return r.choices[0].message.content.strip()
    except Exception:
        return f"คำตอบ: {ent_str}"

def explain_query(ex, show_narration=True):
    model.eval()
    q = ex['question']
    print("="*70); print("【 1. QUERY (คำถามที่เข้ามา) 】"); print("="*70)
    print(f"  {q}\n")

    prompt = build_prompt(q)
    enc = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=False, max_new_tokens=160,
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    sqs = parse_subqueries(txt)
    print("【 2. SUB-QUERIES (policy แบ่งคำถาม) 】"); print("="*70)
    for i, sq in enumerate(sqs, 1): print(f"  SQ{i}: {sq}")
    print()

    pb, pt = predict_types_from_question(q)
    seeds = extract_entities(sqs[0]) if sqs else []
    seed_names = [kg.id_to_name.get(s,'?') for s in seeds]
    envout = env_execute(sqs, ex)
    hop1_names = [kg.id_to_name.get(x,'?') for x in list(envout['hop1'])[:10]]  # ★ list()
    print("【 3. KG CONTEXT (reasoning path บนกราฟ) 】"); print("="*70)
    print(f"  Source entities (seed):  {seed_names}")
    print(f"  Type constraint:         bridge={pb}, target={pt}")
    print(f"  Hop-1 (bridge) sample:   {hop1_names}")
    print(f"  Reasoning path:          {seed_names} --[{pb}]--> bridge --[{pt}]--> answer")
    print()

    retrieved = envout['pred_names']
    raw = generate_answer(q, sqs, retrieved)
    grounded = [p for p in raw if p.lower().strip() in _norm(retrieved)]
    print("【 4. ENTITY SET (คำตอบที่ยึดกราฟ) 】"); print("="*70)
    print(f"  Retrieved ({len(retrieved)}): {retrieved[:10]}{' ...' if len(retrieved)>10 else ''}")
    print(f"  Grounded  ({len(grounded)}): {grounded[:10]}{' ...' if len(grounded)>10 else ''}")
    print()

    ans = None
    if show_narration:
        ans = narrate_answer(q, grounded)
        print("【 5. ANSWER (คำตอบภาษาธรรมชาติ) 】"); print("="*70)
        print(f"  {ans}\n")

    print("【 GOLD (เฉลย) 】")
    print(f"  {ex['gold']}")
    tp = len(_norm(grounded) & _norm(ex['gold']))
    print(f"  ✓ ตรง {tp}/{len(_norm(ex['gold']))} | grounded {len(grounded)} ตัว")
    print("="*70)
    return dict(question=q, sub_queries=sqs, seeds=seed_names,
                reasoning_path=f"{seed_names}->{pb}->{pt}",
                retrieved=retrieved, grounded=grounded,
                narration=ans, gold=ex['gold'])

def export_examples(indices, path='/workspace/outputs/audit_examples.json'):
    examples = [explain_query(test[i]) for i in indices]
    json.dump(examples, open(path,'w'), indent=2, ensure_ascii=False)
    print(f"\n✓ saved {len(examples)} examples to {path}")
    return examples

print("✓ explain_query แก้แล้ว (list() ครอบ hop1)")

In [ ]:
# ดู 1 ตัวอย่าง (audit trail เต็ม)
explain_query(test[0])

# บันทึกหลายตัวอย่างเป็น JSON (สำหรับ paper appendix)
export_examples([0, 1, 2, 5, 10])

In [ ]:
"""Selector — เลือกตัวอย่าง audit trail ให้ครอบคลุมทุกกรณีอย่างเป็นระบบ.

เลือกตาม 3 มิติ:
  1. bucket (≤5, 6-15, 16-50, >50)
  2. relation pattern (ทิศทาง hop)
  3. performance (perfect / partial / error / capped)

ใช้ B11_rldqd_per_q.json (บันทึกจาก run_3tier) — มี f1 รายข้อ
ต้องมีในเคอร์เนล: test, OUTB
"""
import json
from pathlib import Path

def load_per_q():
    p = OUTB / 'B11_rldqd_per_q.json'
    return json.load(open(p))   # list ของ {f1,tp,fp,fn,n_gold,n_pred}

def relation_pattern(ex):
    """ดึง pattern จากคำถาม เช่น gene->disease->drug."""
    import re
    # ใช้ predict_types ถ้ามี หรือ parse คร่าวๆ
    q = ex['question'].lower()
    types_order = []
    for kw, t in [('drug','drug'),('disease','disease'),('gene/protein','gene'),
                  ('gene','gene'),('effect/phenotype','phenotype'),('phenotype','phenotype')]:
        idx = q.find(kw)
        if idx>=0: types_order.append((idx, t))
    types_order.sort()
    seen=[]; 
    for _,t in types_order:
        if t not in seen: seen.append(t)
    return '->'.join(reversed(seen[:3]))   # source->bridge->target

def select_diverse_examples(per_q, n_per_category=2):
    """เลือกตัวแทนแต่ละกรณี คืน dict ของ category -> [indices]."""
    # ผูก per_q กับ test + bucket + pattern
    rows = []
    for i, pq in enumerate(per_q):
        ex = test[i]
        ng = pq['n_gold']
        bucket = '≤5' if ng<=5 else '6-15' if ng<=15 else '16-50' if ng<=50 else '>50'
        rows.append({'idx':i, 'f1':pq['f1'], 'n_gold':ng, 'n_pred':pq['n_pred'],
                     'bucket':bucket, 'pattern':relation_pattern(ex),
                     'question':ex['question'][:55]})

    cats = {}

    # มิติ 1: bucket
    for b in ['≤5','6-15','16-50','>50']:
        items = sorted([r for r in rows if r['bucket']==b],
                       key=lambda r:-r['f1'])
        cats[f'bucket_{b}_best'] = items[:n_per_category]

    # มิติ 3: performance
    perfect = [r for r in rows if r['f1']>=0.99][:n_per_category]
    partial = [r for r in rows if 0.3<=r['f1']<0.7][:n_per_category]
    error   = sorted([r for r in rows if r['f1']<0.15 and r['n_pred']>0],
                     key=lambda r:r['f1'])[:n_per_category]
    capped  = [r for r in rows if r['n_gold']>50][:n_per_category]
    cats['perfect'] = perfect
    cats['partial'] = partial
    cats['error'] = error
    cats['topk_capped'] = capped

    # มิติ 2: pattern diversity (unique patterns)
    seen_pat = set(); pat_examples=[]
    for r in sorted(rows, key=lambda r:-r['f1']):
        if r['pattern'] not in seen_pat:
            seen_pat.add(r['pattern']); pat_examples.append(r)
        if len(pat_examples)>=6: break
    cats['unique_patterns'] = pat_examples

    return cats, rows

def print_selection(cats):
    print("="*72)
    print("ตัวอย่างที่เลือก แบ่งตามกรณี (สำหรับ paper case study + error analysis)")
    print("="*72)
    seen_idx = {}
    for cat, items in cats.items():
        print(f"\n[{cat}]")
        for r in items:
            print(f"  idx={r['idx']:3d} F1={r['f1']:.3f} gold={r['n_gold']:3d} "
                  f"pred={r['n_pred']:3d} [{r['bucket']:5s}] {r['pattern']:20s} | {r['question']}")
            seen_idx[r['idx']] = r
    # รวม index ที่ไม่ซ้ำ
    all_idx = sorted(seen_idx.keys())
    print(f"\n{'='*72}")
    print(f"รวม index ที่ไม่ซ้ำ ({len(all_idx)} ตัว): {all_idx}")
    print("="*72)
    return all_idx

def run_selection(n_per=2):
    per_q = load_per_q()
    cats, rows = select_diverse_examples(per_q, n_per)
    idx = print_selection(cats)
    # สถิติภาพรวม
    print(f"\nสถิติภาพรวม (764 คำถาม):")
    for b in ['≤5','6-15','16-50','>50']:
        bs = [r for r in rows if r['bucket']==b]
        avg = sum(r['f1'] for r in bs)/len(bs) if bs else 0
        print(f"  {b:6s}: n={len(bs):3d}, avg-F1={avg:.3f}")
    return idx

idx = run_selection(2)
export_examples(idx)

In [ ]:
"""B12 Graph-R1 — Phase 1: Multi-turn inference บน PrimeKG (ยังไม่ train).

Adapt จาก Graph-R1 (Luo et al., ICML 2026) มา setup เรา:
  - hypergraph → PrimeKG (entity-relation graph)
  - hyperedge retrieval → kg.get_neighbors (type-aware)
  - multi-turn loop: <think> → <query> → <knowledge> → rethink → <answer>
  - coupled agent (1 policy ทำทั้ง reason + query) ← ต่างจาก BRACED decoupled

Phase 1 = inference เท่านั้น (ใช้ model ที่มี ยังไม่ train Graph-R1 style)
จุดประสงค์: ยืนยัน multi-turn loop ทำงานถูก ก่อนเพิ่ม reward + GRPO (Phase 2-3)

ต้องมีในเคอร์เนล: model, tok, kg, test, extract_entities, run_two_hop,
                  predict_types_from_question, f1_set, generate_answer
"""
import torch, re

# ============================================================
# Prompt — Graph-R1 multi-turn format (ตาม paper Table 1, adapt)
# ============================================================
GRAPHR1_SYSTEM = (
    "You are a helpful biomedical assistant. Answer the question by querying a "
    "knowledge graph. You can query multiple times.\n"
    "First reason inside <think>...</think>. "
    "To query the knowledge graph, write a query inside <query>...</query> "
    "after your thinking. "
    "The retrieved knowledge will be given inside <knowledge>...</knowledge>. "
    "When you have enough information, output the final answer (a list of entity "
    "names) inside <answer>...</answer>.\n"
    "Question: {question}\n"
)

def parse_graphr1_action(text):
    """ดึง action จาก output: think + (query หรือ answer)."""
    think = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    query = re.search(r'<query>(.*?)</query>', text, re.DOTALL)
    answer = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return {
        'think': think.group(1).strip() if think else None,
        'query': query.group(1).strip() if query else None,
        'answer': answer.group(1).strip() if answer else None,
        'well_formed': bool(think) and (bool(query) or bool(answer)),
    }

def graphr1_retrieve(query_str, ex, top_k=15):
    """retrieve จาก PrimeKG ตาม query (adapt จาก hyperedge → neighbor).

    ใช้ type-aware retrieval: extract entity จาก query → get neighbors.
    คืน list ของ (entity_name) + format เป็นข้อความสำหรับ <knowledge>.
    """
    seeds = extract_entities(query_str)
    if not seeds:
        return [], "No matching entities found in the knowledge graph."
    # ดึง neighbor ทุกประเภท (Graph-R1 ไม่ filter type — coupled agent ตัดสินเอง)
    neigh = []
    for s in seeds[:3]:
        for nb in kg.get_neighbors_with_names(s, top_k=top_k):
            if nb not in neigh: neigh.append(nb)
    neigh = neigh[:top_k]
    # format เป็น knowledge text
    if neigh:
        ktext = "; ".join(neigh)
    else:
        ktext = "No neighbors found."
    return neigh, ktext

def graphr1_inference(ex, max_turns=4, verbose=False):
    """Multi-turn inference loop (Graph-R1 style) บน PrimeKG."""
    model.eval()
    prompt = GRAPHR1_SYSTEM.format(question=ex['question'])
    all_retrieved = []   # สะสม entities ที่ retrieve มาทุก turn
    turns = 0
    final_answer = None

    for turn in range(max_turns):
        turns += 1
        enc = tok(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, do_sample=False, max_new_tokens=200,
                                 pad_token_id=tok.pad_token_id)
        gen = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        act = parse_graphr1_action(gen)
        if verbose:
            print(f"  [turn {turn+1}] think={str(act['think'])[:50]}... "
                  f"query={act['query']} answer={'yes' if act['answer'] else 'no'}")

        # ต่อ output เข้า prompt (state update)
        prompt += gen

        if act['answer'] is not None:
            final_answer = act['answer']
            break
        elif act['query'] is not None:
            retrieved, ktext = graphr1_retrieve(act['query'], ex)
            all_retrieved.extend(retrieved)
            prompt += f"\n<knowledge>{ktext}</knowledge>\n"
        else:
            # ไม่มีทั้ง query และ answer → malformed → หยุด
            break

    # parse answer เป็น entity list
    if final_answer:
        pred = [x.strip() for x in re.split(r'[,;\n]', final_answer) if x.strip()]
    else:
        pred = []   # ไม่ตอบ
    # ground answer ให้อยู่ใน retrieved (เหมือน grounded-only ของเรา)
    ctx = {r.lower().strip() for r in all_retrieved}
    grounded = [p for p in pred if p.lower().strip() in ctx]
    return dict(pred=pred, grounded=grounded, retrieved=all_retrieved,
                turns=turns, answered=final_answer is not None)

# ============================================================
# ทดสอบ Phase 1 — รัน 5 ตัวอย่าง ดูว่า loop ทำงาน
# ============================================================
def test_graphr1_phase1(n=5):
    print("="*64)
    print("B12 Graph-R1 Phase 1 — Multi-turn Inference Test")
    print("="*64)
    for i in range(n):
        ex = test[i]
        print(f"\n[{i}] Q: {ex['question'][:60]}")
        r = graphr1_inference(ex, max_turns=4, verbose=True)
        gf1,_,_,_ = f1_set(r['grounded'], ex['gold'])
        rf1,_,_,_ = f1_set(r['retrieved'], ex['gold'])
        print(f"  turns={r['turns']} answered={r['answered']} "
              f"retrieved={len(r['retrieved'])} grounded={len(r['grounded'])}")
        print(f"  grounded-F1={gf1:.3f} (retrieved-F1={rf1:.3f}) gold={len(ex['gold'])}")

print("✓ Graph-R1 Phase 1 ready")
print("  test_graphr1_phase1(5)  — ทดสอบ multi-turn loop 5 ตัว")

In [ ]:
test_graphr1_phase1(5)